# 01 · Data cleaning — Volve daily production

**Goal:** turn the raw Equinor *Daily Production Data* sheet into a clean, analysis-ready table (`data/processed/volve_daily.parquet` + `.csv` for Power BI).

**Cleaning rules decided after profiling the data:**

| # | Issue found | Rule | Why |
|---|---|---|---|
| 1 | Long, inconsistent column names | Rename to short `snake_case` | Readable SQL and Power BI fields |
| 2 | Well names like `15/9-F-12` | Add short name `F-12` | Shorter labels in charts |
| 3 | `WELL_TYPE` changes over time (F-5 was a producer, then an injector) | Use `FLOW_KIND` per day as the well's **role** | It describes what the well actually did that day |
| 4 | Missing volumes on shut-in days | Fill with 0 when the well was not on stream | No flow ⇒ zero volume, not unknown |
| 5 | 4 negative water volumes | Set to 0 and flag | Allocation artefacts; physically impossible |
| 6 | Downhole pressure/temperature = 0 while the well was producing | Set to missing (NaN) and flag | A live well cannot have 0 bar downhole: the gauge failed |
| 7 | 20 days with 25 on-stream hours | Keep, compute efficiency against the real day length | These are the last Sundays of October: daylight-saving change in Norway, so the day really had 25 hours |


In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

RAW = Path("../data/raw/Volve production data.xlsx")
OUT = Path("../data/processed")
OUT.mkdir(parents=True, exist_ok=True)

raw = pd.read_excel(RAW, sheet_name="Daily Production Data")
print(f"{raw.shape[0]:,} rows × {raw.shape[1]} columns")
print(f"Period: {raw['DATEPRD'].min():%Y-%m-%d} → {raw['DATEPRD'].max():%Y-%m-%d}")

15,634 rows × 24 columns
Period: 2007-09-01 → 2016-12-01


## 1 · Profile the raw data

In [2]:
(raw.groupby("NPD_WELL_BORE_NAME")
    .agg(first_day=("DATEPRD", "min"), last_day=("DATEPRD", "max"), days=("DATEPRD", "size"),
         oil_sm3=("BORE_OIL_VOL", "sum"), water_injected_sm3=("BORE_WI_VOL", "sum"))
    .round({"oil_sm3": 0, "water_injected_sm3": 0}))

,first_day,last_day,days,oil_sm3,water_injected_sm3
NPD_WELL_BORE_NAME,,,,,
15/9-F-1 C,2014-04-07,2016-04-21,746,177709.0,0.0
15/9-F-11,2013-07-08,2016-09-17,1165,1147849.0,0.0
15/9-F-12,2008-02-12,2016-09-17,3056,4579610.0,0.0
15/9-F-14,2008-02-12,2016-09-17,3056,3942233.0,0.0
15/9-F-15 D,2014-01-12,2016-09-17,978,148519.0,0.0
15/9-F-4,2007-09-01,2016-12-01,3327,0.0,16240727.0
15/9-F-5,2007-09-01,2016-09-18,3306,41161.0,14089406.0


In [3]:
pd.crosstab(raw["NPD_WELL_BORE_NAME"], [raw["FLOW_KIND"], raw["WELL_TYPE"]])

FLOW_KIND          injection production    
WELL_TYPE                 WI         OP  WI
NPD_WELL_BORE_NAME                         
15/9-F-1 C                 0        744   2
15/9-F-11                  0       1165   0
15/9-F-12                  0       3056   0
15/9-F-14                  0       3056   0
15/9-F-15 D                0        978   0
15/9-F-4                3327          0   0
15/9-F-5                3146        144  16

In [4]:
missing = raw.isna().sum()
missing[missing > 0].sort_values(ascending=False).to_frame("missing_values")

,missing_values
BORE_WI_VOL,9928
AVG_ANNULUS_PRESS,7744
AVG_CHOKE_SIZE_P,6715
AVG_DP_TUBING,6654
AVG_DOWNHOLE_TEMPERATURE,6654
AVG_DOWNHOLE_PRESSURE,6654
AVG_WHT_P,6488
AVG_WHP_P,6479
BORE_GAS_VOL,6473
AVG_CHOKE_UOM,6473


## 2 · Rename columns and add the short well name

In [5]:
COLUMNS = {
    "DATEPRD": "date",
    "NPD_WELL_BORE_NAME": "wellbore",
    "FLOW_KIND": "role",
    "WELL_TYPE": "well_type",
    "ON_STREAM_HRS": "on_stream_hrs",
    "AVG_DOWNHOLE_PRESSURE": "downhole_pressure_bar",
    "AVG_DOWNHOLE_TEMPERATURE": "downhole_temp_c",
    "AVG_DP_TUBING": "dp_tubing_bar",
    "AVG_ANNULUS_PRESS": "annulus_pressure_bar",
    "AVG_CHOKE_SIZE_P": "choke_pct",
    "AVG_WHP_P": "whp_bar",
    "AVG_WHT_P": "wht_c",
    "DP_CHOKE_SIZE": "dp_choke_bar",
    "BORE_OIL_VOL": "oil_sm3",
    "BORE_GAS_VOL": "gas_sm3",
    "BORE_WAT_VOL": "water_sm3",
    "BORE_WI_VOL": "water_injected_sm3",
}
df = raw[list(COLUMNS)].rename(columns=COLUMNS).copy()
df["well"] = df["wellbore"].str.replace("15/9-", "", regex=False).str.replace(" ", "", regex=False)
df["role"] = df["role"].map({"production": "producer", "injection": "injector"})
df = df.sort_values(["well", "date"]).reset_index(drop=True)
df[["date", "wellbore", "well", "role", "on_stream_hrs", "oil_sm3"]].head()

,date,wellbore,well,role,on_stream_hrs,oil_sm3
0,2013-07-08,15/9-F-11,F-11,producer,0.0,0.0
1,2013-07-09,15/9-F-11,F-11,producer,0.0,0.0
2,2013-07-10,15/9-F-11,F-11,producer,0.0,0.0
3,2013-07-11,15/9-F-11,F-11,producer,0.0,0.0
4,2013-07-12,15/9-F-11,F-11,producer,0.0,0.0


## 3 · Time on stream

The 25-hour days all fall on the last Sunday of October, when Norway switches back from summer time. We keep them and measure efficiency against the real length of each day.

In [6]:
over_24 = df.loc[df["on_stream_hrs"] > 24, "date"]
over_24.dt.strftime("%Y-%m-%d (%a)").value_counts().sort_index().to_frame("wells_with_25h")

,wells_with_25h
date,
2008-10-26 (Sun),4
2009-10-25 (Sun),2
2010-10-31 (Sun),3
2013-10-27 (Sun),5
2014-10-26 (Sun),6


In [7]:
df["on_stream_hrs"] = df["on_stream_hrs"].fillna(0)

# Real day length: 25 h on the October DST change, 23 h on the March change, else 24 h
oslo = pd.DatetimeIndex(df["date"].drop_duplicates().sort_values())
day_len = pd.Series(
    ((oslo + pd.Timedelta(days=1)).tz_localize("Europe/Oslo") - oslo.tz_localize("Europe/Oslo"))
    / pd.Timedelta(hours=1),
    index=oslo,
)
df["day_hours"] = df["date"].map(day_len)
df["efficiency"] = (df["on_stream_hrs"] / df["day_hours"]).clip(upper=1)
df["is_on_stream"] = df["on_stream_hrs"] > 0
df["day_hours"].value_counts()

day_hours
24.0    15549
23.0       44
25.0       41
Name: count, dtype: int64

## 4 · Volumes

In [8]:
VOLUMES = ["oil_sm3", "gas_sm3", "water_sm3", "water_injected_sm3"]

df["flag_negative_water"] = df["water_sm3"] < 0
print("Negative water volumes:")
display(df.loc[df["flag_negative_water"], ["date", "well", "water_sm3"]])
df.loc[df["flag_negative_water"], "water_sm3"] = 0

# No flow recorded on shut-in days ⇒ zero, not unknown
df[VOLUMES] = df[VOLUMES].fillna(0)

# Producers do not inject; injectors do not produce (F-5 switched role in 2008)
df.loc[df["role"] == "producer", "water_injected_sm3"] = 0
df.loc[df["role"] == "injector", ["oil_sm3", "gas_sm3", "water_sm3"]] = 0
df.groupby("role")[VOLUMES].sum().round(0)

Negative water volumes:


,date,well,water_sm3
1236,2008-04-23,F-12,-14.19
2756,2012-08-13,F-12,-457.84
4604,2009-03-03,F-14,-0.95
5812,2012-08-13,F-14,-59.19


,oil_sm3,gas_sm3,water_sm3,water_injected_sm3
role,,,,
injector,0.0,0.000000e+00,0.0,30330134.0
producer,10037081.0,1.475370e+09,15319111.0,0.0


## 5 · Sensor readings

A downhole pressure (or temperature) of exactly 0 while the well is flowing is a **failed gauge**, not a real value. Leaving the zeros in would drag every average down, so we turn them into missing values and keep a flag.

In [9]:
SENSORS = ["downhole_pressure_bar", "downhole_temp_c", "dp_tubing_bar", "whp_bar", "wht_c"]

failed = (df[SENSORS] == 0) & df["is_on_stream"].to_numpy()[:, None]
df["flag_gauge_failure"] = failed["downhole_pressure_bar"]
print(failed.sum().to_frame("zero_while_on_stream"))

# Zeros are 'no reading' for these physical measurements
df[SENSORS] = df[SENSORS].mask(df[SENSORS] == 0)

                       zero_while_on_stream
downhole_pressure_bar                  1924
downhole_temp_c                        1924
dp_tubing_bar                             2
whp_bar                                   4
wht_c                                     2


## 6 · Derived well-performance indicators

In [10]:
liquid = df["oil_sm3"] + df["water_sm3"]
df["water_cut"] = np.where(liquid > 0, df["water_sm3"] / liquid, np.nan)
df["gor_sm3_sm3"] = np.where(df["oil_sm3"] > 0, df["gas_sm3"] / df["oil_sm3"], np.nan)
# Rate only for producers with at least 1 h on stream (shorter runs give unrealistic spikes)
producing = (df["role"] == "producer") & (df["on_stream_hrs"] >= 1)
df["oil_rate_sm3_per_hr"] = np.where(producing, df["oil_sm3"] / df["on_stream_hrs"].where(producing), np.nan)
df[["water_cut", "gor_sm3_sm3", "oil_rate_sm3_per_hr"]].describe().round(2)

,water_cut,gor_sm3_sm3,oil_rate_sm3_per_hr
count,8010.00,8008.00,8006.00
mean,0.54,151.96,54.08
std,0.34,16.45,56.78
min,0.00,18.54,0.00
25%,0.16,145.83,11.96
50%,0.64,151.43,32.35
75%,0.85,156.81,71.71
max,1.00,1110.19,299.59


## 7 · Quality checks

In [11]:
checks = {
    "no duplicate well-days": not df.duplicated(["well", "date"]).any(),
    "no negative volumes": (df[VOLUMES] >= 0).all().all(),
    "efficiency between 0 and 1": df["efficiency"].between(0, 1).all(),
    "water cut between 0 and 1": df["water_cut"].dropna().between(0, 1).all(),
    "oil total preserved (Sm3)": round(df["oil_sm3"].sum()) == round(raw["BORE_OIL_VOL"].sum()),
}
for name, ok in checks.items():
    print(("✅" if ok else "❌"), name)
assert all(checks.values())

✅ no duplicate well-days
✅ no negative volumes
✅ efficiency between 0 and 1
✅ water cut between 0 and 1
✅ oil total preserved (Sm3)


## 8 · Save

In [12]:
ORDER = ["date", "well", "wellbore", "role", "well_type", "day_hours", "on_stream_hrs", "efficiency",
         "is_on_stream", "oil_sm3", "gas_sm3", "water_sm3", "water_injected_sm3", "water_cut", "gor_sm3_sm3",
         "oil_rate_sm3_per_hr", "choke_pct", "whp_bar", "wht_c", "downhole_pressure_bar", "downhole_temp_c",
         "dp_tubing_bar", "annulus_pressure_bar", "dp_choke_bar", "flag_negative_water", "flag_gauge_failure"]
clean = df[ORDER]
clean.to_parquet(OUT / "volve_daily.parquet", index=False)
clean.to_csv(OUT / "volve_daily.csv", index=False)
print(f"Saved {len(clean):,} rows × {clean.shape[1]} columns to {OUT}")

Saved 15,634 rows × 26 columns to ..\data\processed


## Summary

- **15,634 well-days**, 7 wellbores (5 producers, 2 water injectors), Sep 2007 → Dec 2016.
- Fixed 4 negative water volumes, turned 1,900+ failed-gauge zeros into missing values, and explained the 25-hour days (DST change).
- Added efficiency, water cut, GOR and oil rate per on-stream hour.
- Oil total preserved exactly after cleaning.

**Next:** `02_analysis.ipynb` — KPIs in SQL (DuckDB) and the production story of the field.